In [ ]:
!pip install -q tifffile imagecodecs

!pip install torchmetrics[detection] pycocotools

import os, random, time
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms.functional as F

In [ ]:
class HistoricBuildingDataset(Dataset):
    def __init__(self, root, class_ids=(1,2,3), image_ext=".tif"):
        self.root = root
        self.images_dir = os.path.join(root, "images")
        self.labels_dir = os.path.join(root, "labels")
        self.class_ids = tuple(class_ids)
        self.image_paths = sorted([
            os.path.join(self.images_dir, f)
            for f in os.listdir(self.images_dir)
            if f.lower().endswith(image_ext)
        ])

        # Check if the file has a matching class id, drop otherwhise
        valid = []
        for p in self.image_paths:
            name = os.path.basename(p)
            has_any = False
            for cid in self.class_ids:
                mp = os.path.join(self.labels_dir, str(cid), name)
                if os.path.exists(mp):
                    has_any = True
                    break # If even one label file exists, keep the image

            if has_any:
                valid.append(p)
        self.image_paths = valid

    def __len__(self):
        return len(self.image_paths)

    def _read_image(self, path):
        img = tifffile.imread(path)
        if img.ndim == 2:
            img = np.stack([img]*3, axis=-1)
        elif img.ndim == 3 and img.shape[-1] == 1:
            img = np.concatenate([img]*3, axis=-1)
        if np.issubdtype(img.dtype, np.floating):
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)
        return img

    def _read_mask(self, path):
        try:
            m = tifffile.imread(path)
            if m.ndim == 3:
                m = m.max(axis=-1)
            return m
        except Exception:
            return None

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = self._read_image(img_path)
        H, W = img.shape[:2]

        boxes = []
        labels = []
        instance_masks = []

        base = os.path.basename(img_path)
        for class_id in self.class_ids:
            mask_path = os.path.join(self.labels_dir, str(class_id), base)
            if not os.path.exists(mask_path):
                continue
            mask_raster = self._read_mask(mask_path)
            if mask_raster is None:
                continue

            unique_ids = np.unique(mask_raster)
            unique_ids = unique_ids[unique_ids != 0]
            for uid in unique_ids:
                single = (mask_raster == uid).astype(np.uint8)
                if single.sum() == 0:
                    continue
                ys, xs = np.where(single)
                ymin, ymax = int(ys.min()), int(ys.max())
                xmin, xmax = int(xs.min()), int(xs.max())
                if xmax <= xmin or ymax <= ymin:
                    continue
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(class_id))
                if single.shape != (H, W):
                    canvas = np.zeros((H, W), dtype=np.uint8)
                    h0, w0 = single.shape
                    canvas[:h0, :w0] = single
                    instance_masks.append(canvas)
                else:
                    instance_masks.append(single)

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            masks = torch.zeros((0, H, W), dtype=torch.uint8)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            masks = torch.as_tensor(np.stack(instance_masks, axis=0), dtype=torch.uint8)

        image = F.to_tensor(img)  # [C,H,W] float32
        target = {"boxes": boxes, "labels": labels, "masks": masks}
        return image, target


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Initialize the Dataset
data_path = '/content/drive/MyDrive/Deep_learning_test'
dataset = HistoricBuildingDataset(data_path)

# Split 
torch.manual_seed(1)
indices = torch.randperm(len(dataset)).tolist()
split_idx = int(len(dataset) * 0.9)

dataset_train = torch.utils.data.Subset(dataset, indices[:split_idx])
dataset_test = torch.utils.data.Subset(dataset, indices[split_idx:])


data_loader = DataLoader(dataset_train, batch_size=4, shuffle=True, collate_fn=collate_fn)

print(f"Loaded {len(dataset_train)} training chips and {len(dataset_test)} validation chips.")

In [ ]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def get_model(num_classes):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # Replace bounding box head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Replace mask head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model(num_classes=4)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

print(f"Model built and loaded onto: {device}")

In [ ]:
#training the model
num_epochs = 10
print("Starting training loop...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    start_time = time.time()

    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Calculate loss
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        epoch_loss += losses.item()

        # Backpropagation (updating the brain)
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{num_epochs} | Average Loss: {epoch_loss/len(data_loader):.4f} | Time: {epoch_time:.0f}s")

torch.save(model.state_dict(), '/content/drive/MyDrive/Deep_learning_model/historic_building_model.pth')
print("Training complete and model successfully saved to Drive!")

Output of previous cell:

Starting training loop...

Epoch 1/10 | Average Loss: 0.5496 | Time: 3856s

Epoch 2/10 | Average Loss: 0.4768 | Time: 1497s

Epoch 3/10 | Average Loss: 0.4501 | Time: 1503s

Epoch 4/10 | Average Loss: 0.4216 | Time: 1503s

Epoch 5/10 | Average Loss: 0.4040 | Time: 1505s

Epoch 6/10 | Average Loss: 0.3862 | Time: 1509s

Epoch 7/10 | Average Loss: 0.3699 | Time: 1512s

Epoch 8/10 | Average Loss: 0.3558 | Time: 1511s

Epoch 9/10 | Average Loss: 0.3423 | Time: 1513s

Epoch 10/10 | Average Loss: 0.3307 | Time: 1516s

Training complete and model successfully saved to Drive!

In [ ]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from google.colab import drive

# Model performance

drive.mount('/content/drive')

validation_data_loader = DataLoader(
    dataset_test, 
    batch_size=4, 
    shuffle=False, 
    collate_fn=collate_fn
)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_model(num_classes=4)
model.load_state_dict(torch.load('/content/drive/MyDrive/Pei_building_model/historic_building_model.pth'))
model.to(device)
model.eval()

metric = MeanAveragePrecision(iou_type="segm")


with torch.no_grad():
    for images, targets in validation_data_loader:
        images = list(img.to(device) for img in images)
        preds = model(images)
        
        preds_formatted = []
        for p in preds:
            preds_formatted.append({
                'boxes': p['boxes'].cpu(),
                'scores': p['scores'].cpu(),
                'labels': p['labels'].cpu(),
                # Model outputs masks as (N, 1, H, W) float probabilities.
                # Squeeze dimension 1 to get (N, H, W) and threshold to binary uint8.
                'masks': (p['masks'].cpu().squeeze(1) > 0.5).to(torch.uint8)
            })

        targets_formatted = []
        for t in targets:
            targets_formatted.append({
                'boxes': t['boxes'].cpu(),
                'labels': t['labels'].cpu(),
                # Dataset already returns masks as (N, H, W) uint8.
                # Direct CPU transfer only; no squeezing or type casting.
                'masks': t['masks'].cpu() 
            })
        
        metric.update(preds_formatted, targets_formatted)

results = metric.compute()
print(f"mAP: {results['map']:.4f}")
print(f"mAP@50: {results['map_50']:.4f}")
print(f"mAP@75: {results['map_75']:.4f}")

In [ ]:
import os
import torch
import torchvision
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


from google.colab import drive
drive.mount('/content/drive')

# 1. Define paths
MODEL_PATH = '/content/drive/MyDrive/Pei_building_model/historic_building_model.pth'
FOLDER_PATH = '/content/drive/MyDrive/princecounty_images'

#confidence level
CONFIDENCE_THRESHOLD = 0.50
CLASS_MAP = {1: "Barn", 2: "House", 3: "Other"}

def get_trained_model(num_classes=4):
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


device = torch.device('cpu')
print("Loading model weights onto CPU...")
model = get_trained_model(num_classes=4)
checkpoint = torch.load(MODEL_PATH, map_location=device)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model.load_state_dict(checkpoint)
model.eval()
print("Model loaded and standing by!")


if not os.path.exists(FOLDER_PATH):
    print(f"[ERROR] Could not find the folder at {FOLDER_PATH}. Please double check the path.")
else:
    valid_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')
    image_files = [f for f in os.listdir(FOLDER_PATH) if f.lower().endswith(valid_extensions)]

    print(f"Found {len(image_files)} sample files to process.\n")

    for filename in image_files:
        full_image_path = os.path.join(FOLDER_PATH, filename)
        print(f"Processing: {filename}...")

        pil_img = Image.open(full_image_path).convert("RGB")
        img_uint8 = np.array(pil_img)

        # Build PyTorch input tensor [C, H, W] scaled to [0, 1]
        input_tensor = torch.as_tensor(img_uint8, dtype=torch.float32).permute(2, 0, 1) / 255.0
        input_tensor = input_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            predictions = model(input_tensor)[0]

        boxes = predictions['boxes'].cpu().numpy()
        labels = predictions['labels'].cpu().numpy()
        scores = predictions['scores'].cpu().numpy()
        masks = predictions['masks'].cpu().squeeze(1).cpu().numpy()

        fig, ax = plt.subplots(1, 1, figsize=(5, 5))
        ax.imshow(img_uint8)

        detection_count = 0
        for i in range(len(scores)):
            if scores[i] < CONFIDENCE_THRESHOLD:
                continue

            detection_count += 1
            xmin, ymin, xmax, ymax = boxes[i]
            label_id = labels[i]

            rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=2, edgecolor='cyan', facecolor='none')
            ax.add_patch(rect)

            mask = masks[i] > 0.5
            mask_overlay = np.zeros((*mask.shape, 4))
            mask_overlay[mask] = [0.0, 1.0, 1.0, 0.3] 
            ax.imshow(mask_overlay)

            label_name = CLASS_MAP.get(label_id, f"Class {label_id}")
            ax.text(xmin, ymin - 4, f"{label_name} ({scores[i]*100:.1f}%)",
                    color='cyan', fontsize=9, bbox=dict(facecolor='black', alpha=0.6, pad=1))

        plt.title(f"File: {filename} | Detections: {detection_count}", fontsize=14)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print("-" * 50)

### Mask R-CNN Detection Results

![Detection 1](images/Screenshot%202026-07-17%20104807.png)

![Detection 2](images/Screenshot%202026-07-17%20104926.png)

![Detection 3](images/Screenshot%202026-07-17%20105017.png)

![Detection 4](images/Screenshot%202026-07-17%20105041.png)

![Detection 5](images/Screenshot%202026-07-17%20105107.png)

![Detection 6](images/Screenshot%202026-07-17%20105518.png)

![Detection 7](images/Screenshot%202026-07-17%20110358.png)


# Run the model over all the `.tif` files and extract the `.geojson` to put back into arcGIS

In [ ]:
!pip install geojson
!pip install pyproj

# county tile prediction

**Last used in Queens county so files point for those results, in practice we would automate it for each county**

In [ ]:
import os
import rasterio
from rasterio.windows import Window
import numpy as np
import math
import shutil
from google.colab import drive

# This piece of code will scan through our giant tif file, chop it into smaller 256x256 squares and filter the tiles that are not useful for us
drive.mount('/content/drive/')

INPUT_TIFF = "/content/drive/MyDrive/queens_test/queens_final.tif"
LOCAL_TMP_DIR = "/content/queens_county_chips/"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/queens_test/queens_images_overlap"

TILE_SIZE = 256
OVERLAP_PERCENT = 0.20  
VARIANCE_THRESHOLD = 16.3
BLACK_CLUSTER_THRESHOLD = 0.25

STRIDE = int(TILE_SIZE * (1.0 - OVERLAP_PERCENT))

os.makedirs(LOCAL_TMP_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

with rasterio.open(INPUT_TIFF) as src:
    width = src.width
    height = src.height

    x_coords = list(range(0, width - TILE_SIZE + 1, STRIDE))
    y_coords = list(range(0, height - TILE_SIZE + 1, STRIDE))

    if x_coords[-1] + TILE_SIZE < width:
        x_coords.append(width - TILE_SIZE)
    if y_coords[-1] + TILE_SIZE < height:
        y_coords.append(height - TILE_SIZE)

    expected_tiles = len(x_coords) * len(y_coords)

    print(f"Starting Overlapping Production Run...")
    print(f"Tile Size: {TILE_SIZE} | Overlap: {OVERLAP_PERCENT*100}% | Calculated Stride: {STRIDE} pixels")
    print(f"Target Threshold: {VARIANCE_THRESHOLD} | Maximum Possible Yield: ~{expected_tiles} tiles")
    print("Slicing and filtering georeferenced land tiles locally...")

    saved_count = 0

    for row_start in y_coords:
        for col_start in x_coords:
            # Create a 256x256 window stepping by the Stride size
            window = Window(col_start, row_start, TILE_SIZE, TILE_SIZE)
            data = src.read(window=window)

            # 1. Base NoData Check
            if np.count_nonzero(data) < (TILE_SIZE * TILE_SIZE * 0.05):
                continue

            # 2. Black Cluster Check
            if (np.sum(data == 0) / data.size) > BLACK_CLUSTER_THRESHOLD:
                continue

            # 3. Variance Filter
            if float(np.std(data)) < VARIANCE_THRESHOLD:
                continue

            transform = src.window_transform(window)
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": window.height,
                "width": window.width,
                "transform": transform
            })

            # Save with distinctive naming convention indicating pixel coordinates
            out_filepath = os.path.join(LOCAL_TMP_DIR, f"kings_tile_y{row_start}_x{col_start}.tif")
            with rasterio.open(out_filepath, "w", **out_meta) as dest:
                dest.write(data)

            saved_count += 1
            if saved_count % 5000 == 0:
                print(f"Successfully generated {saved_count} true land tiles...")

print(f"\nTiling complete! Final land chip count: {saved_count}")
print("Compressing master archive and exporting straight to Google Drive. Please wait...")

zip_target_path = os.path.join(DRIVE_OUTPUT_DIR, "queens_chips_production")
shutil.make_archive(zip_target_path, 'zip', LOCAL_TMP_DIR)

print(f"Success! Master archive saved to: {zip_target_path}.zip")

In [ ]:
import os
import shutil
import rasterio
from shapely.geometry import Polygon, mapping
from skimage.measure import find_contours
import geojson
import pyproj
import numpy as np
import torch
from PIL import Image

model.eval()

#
BATCH_SIZE = 64 
CONFIDENCE_THRESHOLD = 0.75
CHUNK_SIZE = 2048 

CLASS_MAP = {
    1: "Barn",
    2: "House",
    3: "Other buildings"
}

ZIP_PATH = "/content/drive/MyDrive/queens_test/queens_images_overlap/queens_chips_production.zip"
LOCAL_UNZIP_DIR = "/content/queens_county_chips/"

OUTPUT_FOLDER = "/content/drive/MyDrive/Pei_building_model"
CHECKPOINT_DIR = os.path.join(OUTPUT_FOLDER, "queens_checkpoints")
OUTPUT_FILENAME = "queenscounty.geojson"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


if not os.path.exists(LOCAL_UNZIP_DIR) or len(os.listdir(LOCAL_UNZIP_DIR)) == 0:
    if not os.path.exists(ZIP_PATH):
        raise FileNotFoundError(f"Could not find the zip file at {ZIP_PATH}.")
    print("Extracting production archive to local environment...")
    os.makedirs(LOCAL_UNZIP_DIR, exist_ok=True)
    shutil.unpack_archive(ZIP_PATH, LOCAL_UNZIP_DIR, 'zip')
    print("Unzip complete!")
else:
    print("Local chips already present. Skipping extraction step.")

valid_extensions = ('.tif', '.tiff')
image_files = sorted([f for f in os.listdir(LOCAL_UNZIP_DIR) if f.lower().endswith(valid_extensions)])
total_files = len(image_files)
total_chunks = (total_files + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Total tiles to process: {total_files} ({total_chunks} checkpoint chunks)")

# Build Coordinate Transformer once
if total_files > 0:
    first_file = os.path.join(LOCAL_UNZIP_DIR, image_files[0])
    with rasterio.open(first_file) as src:
        transformer = pyproj.Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True)

for chunk_idx in range(total_chunks):
    chunk_filename = f"queens_chunk_{chunk_idx}.geojson"
    chunk_path = os.path.join(CHECKPOINT_DIR, chunk_filename)

    # Skip chunk if it was already successfully computed and saved to Drive
    if os.path.exists(chunk_path):
        print(f"--> Chunk {chunk_idx + 1}/{total_chunks} found on Drive. Skipping...")
        continue

    start_idx = chunk_idx * CHUNK_SIZE
    end_idx = min(start_idx + CHUNK_SIZE, total_files)
    chunk_files = image_files[start_idx:end_idx]

    print(f"\nProcessing Chunk {chunk_idx + 1}/{total_chunks} (Tiles {start_idx} to {end_idx})...")
    chunk_features = []

    batch_tensors = []
    batch_metadata = []

    for idx, filename in enumerate(chunk_files):
        full_image_path = os.path.join(LOCAL_UNZIP_DIR, filename)

        with rasterio.open(full_image_path) as src:
            if not src.crs or not src.transform:
                continue
            img_transform = src.transform

        pil_img = Image.open(full_image_path).convert("RGB")
        img_uint8 = np.array(pil_img)
        input_tensor = torch.as_tensor(img_uint8, dtype=torch.float32).permute(2, 0, 1) / 255.0

        batch_tensors.append(input_tensor)
        batch_metadata.append({"filename": filename, "transform": img_transform})

        # Process complete batch or end of chunk
        if len(batch_tensors) == BATCH_SIZE or idx == len(chunk_files) - 1:
            with torch.no_grad():
                batch = [t.to(device) for t in batch_tensors]
                batch_predictions = model(batch)

            for batch_idx, predictions in enumerate(batch_predictions):
                meta = batch_metadata[batch_idx]
                current_filename = meta["filename"]
                current_transform = meta["transform"]

                masks = predictions['masks'].cpu().squeeze(1).numpy()
                labels = predictions['labels'].cpu().numpy()
                scores = predictions['scores'].cpu().numpy()

                for j in range(len(scores)):
                    if scores[j] < CONFIDENCE_THRESHOLD or labels[j] == 0:
                        continue

                    label_name = CLASS_MAP.get(labels[j], f"Class {labels[j]}")
                    mask = masks[j] > 0.5
                    contours = find_contours(mask, 0.5)

                    if not contours:
                        continue

                    main_contour = contours[0]
                    geo_coords_wgs84 = []
                    coords_orig_crs = []

                    for row, col in main_contour:
                        x_orig_crs, y_orig_crs = current_transform * (col, row)
                        if not (np.isfinite(x_orig_crs) and np.isfinite(y_orig_crs)):
                            continue
                        coords_orig_crs.append((x_orig_crs, y_orig_crs))

                        lon, lat = transformer.transform(x_orig_crs, y_orig_crs)
                        if not (np.isfinite(lon) and np.isfinite(lat)):
                            continue
                        geo_coords_wgs84.append((lon, lat))

                    if geo_coords_wgs84 and len(geo_coords_wgs84) >= 3:
                        polygon = Polygon(geo_coords_wgs84)
                        if not polygon.is_valid:
                            polygon = polygon.buffer(0)
                            if not polygon.is_valid:
                                continue

                        area_sq_units = 0.0
                        if len(coords_orig_crs) >= 3:
                            orig_polygon = Polygon(coords_orig_crs)
                            if not orig_polygon.is_valid:
                                orig_polygon = orig_polygon.buffer(0)
                            area_sq_units = orig_polygon.area

                        feature = geojson.Feature(
                            geometry=mapping(polygon),
                            properties={
                                "image_file": current_filename,
                                "label": label_name,
                                "score": float(scores[j]),
                                "area": round(area_sq_units, 2)
                            }
                        )
                        chunk_features.append(feature)

            batch_tensors = []
            batch_metadata = []

    # Write out this individual chunk file directly to Google Drive
    chunk_collection = geojson.FeatureCollection(chunk_features)
    with open(chunk_path, 'w') as f:
        geojson.dump(chunk_collection, f)
    print(f"Saved chunk {chunk_idx + 1} ({len(chunk_features)} detections) to Drive.")


print("\nAll chunks generated! Loading into GeoPandas for spatial duplicate filtering...")
import geopandas as gpd
from shapely.geometry import shape

compiled_features = []
for chunk_idx in range(total_chunks):
    chunk_path = os.path.join(CHECKPOINT_DIR, f"queens_chunk_{chunk_idx}.geojson")
    with open(chunk_path, 'r') as f:
        data = geojson.load(f)
        if "features" in data:
            compiled_features.extend(data["features"])

# Convert raw GeoJSON features into a spatial GeoDataFrame
gdf = gpd.GeoDataFrame.from_features(compiled_features, crs="EPSG:4326")

if not gdf.empty:
    print(f"Total raw shapes detected: {len(gdf)}. Running Spatial Non-Maximum Suppression...")

    # Sort by confidence score descending so best predictions take priority
    gdf = gdf.sort_values(by="score", ascending=False)

    # Project to metric CRS momentarily to calculate precise real-world overlaps
    gdf_metric = gdf.to_crs(epsg=2954)

    keep_indices = []
    geometries = gdf_metric.geometry.values
    labels = gdf_metric['label'].values

    # Quick spatial loop to remove duplicates
    for idx in range(len(gdf_metric)):
        current_geom = geometries[idx]
        current_label = labels[idx]

        # Check if this shape heavily overlaps with any shape we already decided to keep
        is_duplicate = False
        for keep_idx in keep_indices:
            if current_label == labels[keep_idx]:
                intersection_area = current_geom.intersection(geometries[keep_idx]).area
                union_area = current_geom.union(geometries[keep_idx]).area
                iou = intersection_area / union_area if union_area > 0 else 0

                # if they overlap by more than 35%, it's the same building caught in two tiles
                if iou > 0.35:
                    is_duplicate = True
                    break

        if not is_duplicate:
            keep_indices.append(idx)

    # Filter out the duplicates
    gdf_cleaned = gdf.iloc[keep_indices]
    print(f"Filtered out {len(gdf) - len(gdf_cleaned)} overlapping duplicates!")

    # Save the clean, perfect dataset straight to your Drive
    master_output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILENAME)
    gdf_cleaned.to_file(master_output_path, driver="GeoJSON")
    print(f"Success! Master cleaned dataset saved to: {master_output_path}")
    print(f"Total unique properties mapped across Queens County: {len(gdf_cleaned)}")
else:
    print("No features detected across any chunks.")

In [ ]:
import os
import geojson
import geopandas as gpd

OUTPUT_FOLDER = "/content/drive/MyDrive/Pei_building_model"
CHECKPOINT_DIR = os.path.join(OUTPUT_FOLDER, "queens_checkpoints")
OUTPUT_FILENAME = "queenscountycorrect.geojson"

compiled_features = []

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"Cannot find checkpoint directory: {CHECKPOINT_DIR}")

chunk_files = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith("queens_chunk_") and f.endswith(".geojson")]

for chunk_file in chunk_files:
    chunk_path = os.path.join(CHECKPOINT_DIR, chunk_file)
    with open(chunk_path, 'r') as f:
        data = geojson.load(f)
        if "features" in data:
            compiled_features.extend(data["features"])

gdf = gpd.GeoDataFrame.from_features(compiled_features, crs="EPSG:4326")

if not gdf.empty:
    gdf_metric = gdf.to_crs(epsg=2954)

    gdf_metric['geometry'] = gdf_metric['geometry'].buffer(0.5)
    dissolved = gdf_metric.dissolve(by='label', aggfunc={'score': 'max'})
    exploded = dissolved.explode(index_parts=True).reset_index()
    exploded['geometry'] = exploded['geometry'].buffer(-0.5)

    exploded = exploded[~exploded.is_empty & exploded.is_valid].copy()
    exploded['area'] = exploded['geometry'].area.round(2)

    exploded['uid'] = range(len(exploded))
    overlaps = gpd.sjoin(exploded, exploded, predicate='intersects')
    overlaps = overlaps[overlaps['uid_left'] != overlaps['uid_right']]
    
    to_drop = set()
    
    for idx, row in overlaps.iterrows():
        id_1 = row['uid_left']
        id_2 = row['uid_right']
        
        if id_1 in to_drop or id_2 in to_drop:
            continue
            
        geom_1 = exploded.loc[exploded['uid'] == id_1, 'geometry'].values[0]
        geom_2 = exploded.loc[exploded['uid'] == id_2, 'geometry'].values[0]
        
        min_area = min(geom_1.area, geom_2.area)
        if min_area == 0:
            continue
            
        overlap_ratio = geom_1.intersection(geom_2).area / min_area
        
        if overlap_ratio > 0.30:
            score_1 = exploded.loc[exploded['uid'] == id_1, 'score'].values[0]
            score_2 = exploded.loc[exploded['uid'] == id_2, 'score'].values[0]
            
            if score_1 >= score_2:
                to_drop.add(id_2)
            else:
                to_drop.add(id_1)

    exploded = exploded[~exploded['uid'].isin(to_drop)].drop(columns=['uid'])

    final_gdf = exploded.to_crs(epsg=4326)
    master_output_path = os.path.join(OUTPUT_FOLDER, OUTPUT_FILENAME)
    final_gdf.to_file(master_output_path, driver="GeoJSON")
